In [66]:
import pandas as pd
import numpy as np

In [67]:
orders = pd.read_csv(
    "../data/processed/delivered_orders_clean.csv"
)

customers = pd.read_csv(
    "../data/processed/customers_clean.csv"
)

order_items = pd.read_csv(
    "../data/processed/order_items_clean.csv"
)

payments = pd.read_csv(
    "../data/processed/payments_clean.csv"
)

products = pd.read_csv(
    "../data/processed/products_clean.csv"
)

sellers = pd.read_csv(
    "../data/processed/sellers_clean.csv"
)

In [68]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [69]:
orders.merge(order_items)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110168,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,22.193727,1,f1d4ce8c6dd66c47bbaa8c6781c2a923,1f9ab4708f3056ede07124aad39a2554,2018-02-12 13:10:37,174.90,20.10
110169,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,24.859421,1,b80910977a37536adeddd63663f916ad,d50d79cb34e38265a8649c383dcffd48,2017-09-05 15:04:16,205.99,65.02
110170,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,17.086424,1,d1c427060a0f73f6b889a5c7c61f2ac4,a1043bafd471dff536d0c462352beb48,2018-01-12 21:36:21,179.99,40.59
110171,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,17.086424,2,d1c427060a0f73f6b889a5c7c61f2ac4,a1043bafd471dff536d0c462352beb48,2018-01-12 21:36:21,179.99,40.59


In [70]:
order_item_features = (
    order_items
    .groupby("order_id")
    .agg(
        number_of_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        average_item_price=("price", "mean"),
        number_of_unique_products=("product_id", "nunique"),
        number_of_unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

In [71]:
order_item_features.head()

,order_id,number_of_items,total_price,total_freight,average_item_price,number_of_unique_products,number_of_unique_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,58.90,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,239.90,1,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,199.00,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,12.99,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,199.90,1,1


In [72]:
order_products = order_items.merge(
    products[
        [
            "product_id",
            "product_category_name_english",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ],
    on="product_id",
    how="left"
)

In [73]:
order_products.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,furniture_decor,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumery,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,garden_tools,3750.0,35.0,40.0,30.0


In [74]:
product_features = (
    order_products
    .groupby("order_id")
    .agg(
        average_product_weight_g=(
            "product_weight_g",
            "mean"
        ),

        max_product_weight_g=(
            "product_weight_g",
            "max"
        ),

        average_product_length_cm=(
            "product_length_cm",
            "mean"
        ),

        average_product_height_cm=(
            "product_height_cm",
            "mean"
        ),

        average_product_width_cm=(
            "product_width_cm",
            "mean"
        ),

        number_of_product_categories=(
            "product_category_name_english",
            "nunique"
        )
    )
    .reset_index()
)

In [75]:
master = orders.merge(
    customers[
        [
            "customer_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

In [76]:
master.shape

(96455, 12)

In [77]:
master = master.merge(
    order_item_features,
    on="order_id",
    how="left"
)

In [78]:
master = master.merge(
    product_features,
    on="order_id",
    how="left"
)

In [79]:
order_sellers = order_items.merge(
    sellers[
        [
            "seller_id",
            "seller_city",
            "seller_state",
            "seller_zip_code_prefix"
        ]
    ],
    on="seller_id",
    how="left"
)

In [80]:
seller_features = (
    order_sellers
    .groupby("order_id")
    .agg(
        number_of_sellers=(
            "seller_id",
            "nunique"
        ),

        seller_state_count=(
            "seller_state",
            "nunique"
        )
    )
    .reset_index()
)

In [81]:
master = master.merge(
    seller_features,
    on="order_id",
    how="left"
)

In [82]:
primary_seller = (
    order_sellers
    .sort_values(
        ["order_id", "order_item_id"]
    )
    .drop_duplicates(
        "order_id"
    )
)

In [83]:
primary_seller = primary_seller[
    [
        "order_id",
        "seller_id",
        "seller_city",
        "seller_state",
        "seller_zip_code_prefix"
    ]
]

In [84]:
master = master.merge(
    primary_seller,
    on="order_id",
    how="left"
)

In [85]:
payment_features = (
    payments
    .groupby("order_id")
    .agg(
        total_payment_value=(
            "payment_value",
            "sum"
        ),

        number_of_payment_records=(
            "payment_sequential",
            "count"
        ),

        average_payment_value=(
            "payment_value",
            "mean"
        ),

        max_installments=(
            "payment_installments",
            "max"
        )
    )
    .reset_index()
)

In [86]:
master = master.merge(
    payment_features,
    on="order_id",
    how="left"
)

In [87]:
master["order_purchase_timestamp"] = pd.to_datetime(
    master["order_purchase_timestamp"]
)

In [88]:
master["order_hour"] = (
    master["order_purchase_timestamp"]
    .dt.hour
)

In [89]:
master["order_day_of_week"] = (
    master["order_purchase_timestamp"]
    .dt.dayofweek
)

In [90]:
master["order_month"] = (
    master["order_purchase_timestamp"]
    .dt.month
)

In [91]:
master["is_weekend"] = (
    master["order_day_of_week"] >= 5
).astype(int)

In [92]:
master["delivery_days"] = (
    pd.to_datetime(
        master["order_delivered_customer_date"]
    )
    -
    pd.to_datetime(
        master["order_purchase_timestamp"]
    )
).dt.total_seconds() / (60 * 60 * 24)

In [93]:
master["delivery_days"].describe()

count    96455.000000
mean        12.557693
std          9.545982
min          0.533414
25%          6.765498
50%         10.216227
75%         15.720145
max        209.628611
Name: delivery_days, dtype: float64

In [94]:
master["order_estimated_delivery_date"] = pd.to_datetime(
    master["order_estimated_delivery_date"]
)

master["order_delivered_customer_date"] = pd.to_datetime(
    master["order_delivered_customer_date"]
)

In [95]:
master["delivery_delay_days"] = (
    master["order_delivered_customer_date"]
    -
    master["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

In [96]:
master["late_delivery"] = (
    master["delivery_delay_days"] > 0
).astype(int)

In [97]:
master.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'number_of_items',
 'total_price',
 'total_freight',
 'average_item_price',
 'number_of_unique_products',
 'number_of_unique_sellers',
 'average_product_weight_g',
 'max_product_weight_g',
 'average_product_length_cm',
 'average_product_height_cm',
 'average_product_width_cm',
 'number_of_product_categories',
 'number_of_sellers',
 'seller_state_count',
 'seller_id',
 'seller_city',
 'seller_state',
 'seller_zip_code_prefix',
 'total_payment_value',
 'number_of_payment_records',
 'average_payment_value',
 'max_installments',
 'order_hour',
 'order_day_of_week',
 'order_month',
 'is_weekend',
 'delivery_delay_days',
 'late_delivery']

In [98]:
leakage_columns = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "delivery_delay_days",
    "late_delivery"
]

In [99]:
regression_data = master.drop(
    columns=[
        "order_delivered_customer_date",
        "order_delivered_carrier_date",
        "delivery_delay_days",
        "late_delivery"
    ],
    errors="ignore"
).copy()

In [100]:
classification_data = master.drop(
    columns=[
        "order_delivered_customer_date",
        "order_delivered_carrier_date",
        "delivery_delay_days",
        "delivery_days"
    ],
    errors="ignore"
).copy()

In [101]:
customer_order_count = (
    orders
    .groupby("customer_id")
    .size()
    .reset_index(
        name="customer_order_count"
    )
)

In [102]:
master = master.merge(
    customer_order_count,
    on="customer_id",
    how="left"
)

In [103]:
master["same_state"] = (
    master["seller_state"]
    ==
    master["customer_state"]
).astype(int)

In [104]:
master["average_product_volume_cm3"] = (
    master["average_product_length_cm"]
    *
    master["average_product_height_cm"]
    *
    master["average_product_width_cm"]
)

In [105]:
master["freight_ratio"] = (
    master["total_freight"]
    /
    master["total_price"].replace(0, np.nan)
)

In [106]:
master["freight_ratio"].describe()

count    96455.000000
mean         0.308398
std          0.311637
min          0.000000
25%          0.132017
50%          0.224374
75%          0.380585
max         21.447059
Name: freight_ratio, dtype: float64

In [107]:
master.info()

<class 'pandas.DataFrame'>
RangeIndex: 96455 entries, 0 to 96454
Data columns (total 44 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96455 non-null  str           
 1   customer_id                    96455 non-null  str           
 2   order_status                   96455 non-null  str           
 3   order_purchase_timestamp       96455 non-null  datetime64[us]
 4   order_approved_at              96455 non-null  datetime64[us]
 5   order_delivered_carrier_date   96455 non-null  datetime64[us]
 6   order_delivered_customer_date  96455 non-null  datetime64[us]
 7   order_estimated_delivery_date  96455 non-null  datetime64[us]
 8   delivery_days                  96455 non-null  float64       
 9   customer_zip_code_prefix       96455 non-null  int64         
 10  customer_city                  96455 non-null  str           
 11  customer_state            

In [108]:
master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,customer_zip_code_prefix,...,order_hour,order_day_of_week,order_month,is_weekend,delivery_delay_days,late_delivery,customer_order_count,same_state,average_product_volume_cm3,freight_ratio
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,3149,...,10,0,10,0,-7.107488,0,1,1,1976.0,0.290764
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,47813,...,20,1,7,0,-5.355729,0,1,0,4693.0,0.191744
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,75265,...,8,2,8,0,-17.245498,0,1,0,9576.0,0.120200
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,59296,...,19,5,11,1,-12.980069,0,1,0,6000.0,0.604444
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,9195,...,21,1,2,0,-9.238171,0,1,1,11475.0,0.438191


In [109]:
master.shape

(96455, 44)

In [110]:
master.isnull().sum().sort_values(
    ascending=False
).head(20)

average_product_length_cm        16
average_product_height_cm        16
max_product_weight_g             16
average_product_weight_g         16
average_product_volume_cm3       16
average_product_width_cm         16
max_installments                  1
number_of_payment_records         1
average_payment_value             1
total_payment_value               1
order_id                          0
customer_id                       0
customer_state                    0
customer_city                     0
customer_zip_code_prefix          0
delivery_days                     0
order_estimated_delivery_date     0
order_delivered_customer_date     0
order_delivered_carrier_date      0
order_approved_at                 0
dtype: int64

In [111]:
print("Rows:", len(master))

print(
    "Unique orders:",
    master["order_id"].nunique()
)

Rows: 96455
Unique orders: 96455


In [112]:
master["order_id"].duplicated().sum()

np.int64(0)

In [113]:
regression_data.to_csv(
    "../data/processed/regression_dataset.csv",
    index=False
)

In [114]:
classification_data.to_csv(
    "../data/processed/classification_dataset.csv",
    index=False
)

In [115]:
master.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)